# FDSN Seismology Feeder (Fabric Notebook)

Polls the pre-configured FDSN Event node catalog and emits CloudEvents to the bound Fabric Event Stream.


In [ ]:
# === PARAMETERS (overwritten by deploy-feeder-notebook.ps1) ===
EVENTSTREAM_NAME = "fdsn-seismology-ingest"  # Fabric Event Stream item name in this workspace
STATE_FILE       = "/lakehouse/default/Files/feeder-state/fdsn-seismology/state.json"
POLLING_INTERVAL = 60                        # seconds between polls (earthquake event polling)
RUN_DURATION     = 3300                      # seconds to run before exiting (55 min; scheduler restarts)
ONCE_MODE        = False                     # False = continuous polling loop
WORKSPACE_ID     = ""                        # filled in by deploy script (optional; falls back to runtime context)
FDSN_BASE_URL    = ""                        # FDSN FDSNWS event base URL (injected at deploy time)


In [ ]:
import subprocess, sys, glob as _glob, importlib

_wheels = _glob.glob('/lakehouse/default/Files/wheels/fdsn-seismology/*.whl')
# Install local wheels first (force-reinstall to avoid stale cached versions in Fabric env)
if _wheels:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps'] + _wheels
    )
# Install all PyPI deps including cloudevents and confluent_kafka together
# (cloudevents.kafka requires confluent_kafka to be present when first imported;
#  installing them together prevents Python from caching a failed import of cloudevents.kafka)
_pypi_deps = ['avro>=1.11.3', 'dataclasses_json>=0.6.7', 'confluent_kafka>=2.5.3', 'cloudevents>=1.11.0,<2.0.0', 'pip', 'install', '--force-reinstall', '--upgrade']
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall'] + _pypi_deps
)
# Clear any cached import failures (import system caches failures; stale cache causes ModuleNotFoundError)
for _key in list(sys.modules.keys()):
    if _key.startswith('cloudevents'):
        del sys.modules[_key]
importlib.invalidate_caches()
print(f'Installed {len(_wheels)} wheel(s) + {len(_pypi_deps)} PyPI deps')


In [ ]:
import json, os, requests, time

FABRIC_API = 'https://api.fabric.microsoft.com/v1'

def _get_pbi_token():
    try:
        import notebookutils
        return notebookutils.credentials.getToken('pbi')
    except Exception:
        from notebookutils import mssparkutils
        return mssparkutils.credentials.getToken('pbi')

def _resolve_workspace_id(explicit: str) -> str:
    if explicit:
        return explicit
    try:
        import notebookutils
        return notebookutils.runtime.context['currentWorkspaceId']
    except Exception:
        from notebookutils import mssparkutils
        return json.loads(mssparkutils.runtime.context)['currentWorkspaceId']

def lookup_es_connection_string(workspace_id: str, eventstream_name: str) -> str:
    headers = {'Authorization': f'Bearer {_get_pbi_token()}'}
    eventstreams = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams', headers=headers, timeout=30)
    eventstreams.raise_for_status()
    eventstream = next((item for item in eventstreams.json().get('value', []) if item['displayName'] == eventstream_name), None)
    if not eventstream:
        raise RuntimeError(f"Event Stream '{eventstream_name}' not found in workspace {workspace_id}.")
    topology = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{eventstream["id"]}/topology', headers=headers, timeout=30)
    topology.raise_for_status()
    source = next((item for item in topology.json().get('sources', []) if item.get('type') == 'CustomEndpoint'), None)
    if not source:
        raise RuntimeError('Event Stream has no CustomEndpoint source.')
    last_err = None
    for attempt in range(5):
        try:
            connection = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{eventstream["id"]}/sources/{source["id"]}/connection', headers=headers, timeout=30)
            connection.raise_for_status()
            connection_string = connection.json().get('accessKeys', {}).get('primaryConnectionString')
            if connection_string:
                return connection_string
            last_err = 'empty primaryConnectionString'
        except BaseException as exc:
            last_err = str(exc)
        time.sleep(min(15, 3 * (attempt + 1)))
    raise RuntimeError(f'Could not retrieve connection string after 5 attempts: {last_err}')

workspace_id = _resolve_workspace_id(WORKSPACE_ID)
connection_string = lookup_es_connection_string(workspace_id, EVENTSTREAM_NAME)
os.environ['CONNECTION_STRING'] = connection_string
print(f'Workspace: {workspace_id}')
print(f'Event Stream: {EVENTSTREAM_NAME}')
print(f'Connection ready: length={len(connection_string)}')


In [ ]:
import os, pathlib, sys, traceback, datetime

LOG_PATH = "/lakehouse/default/Files/feeder-state/fdsn-seismology/last-run.log"
pathlib.Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)

def _log(msg):
    line = f"[{datetime.datetime.utcnow().isoformat()}Z] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + '\n')

with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write('')

try:
    _log('Starting feeder (continuous mode)')
    pathlib.Path(STATE_FILE).parent.mkdir(parents=True, exist_ok=True)
    os.environ['STATE_FILE']       = STATE_FILE
    os.environ['POLLING_INTERVAL'] = str(POLLING_INTERVAL)
    os.environ['ONCE_MODE']        = 'true' if ONCE_MODE else 'false'
    if FDSN_BASE_URL:
        os.environ['FDSN_BASE_URL'] = FDSN_BASE_URL
    _log(f'CONNECTION_STRING present: {bool(os.environ.get("CONNECTION_STRING"))}')

    _log('Importing fdsn_seismology package...')
    from fdsn_seismology import app as feeder
    _log(f'Imported feeder from: {feeder.__file__}')

    argv_backup = sys.argv
    try:
        sys.argv = ['fdsn-seismology-kafka', 'feed']
        _log(f'Running feeder.main() with argv={sys.argv}, duration={RUN_DURATION}s')
        import threading
        _err = []
        def _worker():
            try:
                feeder.main()
            except BaseException as e:
                _err.append(e)
        t = threading.Thread(target=_worker, daemon=True)
        t.start()
        t.join(timeout=RUN_DURATION)
        if t.is_alive():
            _log(f'Run duration {RUN_DURATION}s reached; exiting cleanly.')
        elif _err:
            raise _err[0]
    finally:
        sys.argv = argv_backup
    _log('Cycle complete.')
    try:
        import notebookutils
        notebookutils.notebook.exit('OK')
    except Exception:
        pass
except BaseException as exc:
    tb = traceback.format_exc()
    _log(f'FAILED: {exc}\n{tb}')
    try:
        import notebookutils
        notebookutils.notebook.exit(f'FAIL: {exc}')
    except Exception:
        pass
    raise
